# Suite P3DEF — Python 3 definitions, read from the store

The names a Python 3 module binds, in the scopes Python gives them (`Python3.Definitions.define(store, path)`), read
from the module the store holds and computed on demand, nothing stored: one entity per name per scope, with every place
that binds it, and lookup outward through enclosing functions to the module, past class bodies. A place is an `Anchor`,
plain data that outlives the transaction it was read in: its statement's place below the module, the statement's
property and the term's path within it, written as text (`body[1].body[0]:value.args[0]`), by which each case names the
places it checks. The whole tree is values: a module holds its statements as one value, its `body`, and each statement
or clause holds those of its blocks as its own properties, so a place is found by where it is, not by an object
reference. Each case parses a module into one store (`Python3.Parse.parse`), defines it, and checks the entities, their
declarations, and what names refer to. The cases port PYDEF's.

In [ ]:
import itertools
import re
import time
from pathlib import Path

from mbse.Expressions.Framework import Objects, Terms
from mbse.Expressions.Python.Patterns import Expressions as Patterns
from mbse.Expressions.Python.Python3 import Expressions as P
from mbse.Programs import Python3
from mbse.Programs.Framework.Model import WORKSPACE
from mbse.Programs.Python3 import Definitions as D, Parse
from mbse.Schemas.Framework import Schemas
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Python3.store()
PATHS = (f"case{n}.py" for n in itertools.count())


def define(text):
    """The program of the module `text` parses to, written in the store at a path of its own."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return D.define(STORE, path)


def listing(program):
    return [(e.kind, e.qualified_name(), len(e.declarations)) for e in program.entities()]


def refers(program, at):
    return [e.qualified_name() for e in D.referents(program, at)]


def named(program, spelling):
    """The places a name spelled `spelling` is written, as text, in source order."""
    return [str(at) for at in program.named(spelling)]

## P3DEF-01 · Names bound in a scope are local to it, one entity per name, whatever binds it

In [ ]:
program = define("""
x = 1
x += 2
y: int
a, (b, *c) = [d] = e = f()
for i in range(3):
    del i
with open() as (g, h.attribute), open() as k[0]:
    pass
try:
    pass
except E as err:
    pass
import os.path, sys as system
from . import mod as alias, other
from m import *
def function(): pass
class Klass: pass
""")
assert listing(program) == [
    ("variable", "x", 2), ("variable", "y", 1), ("variable", "a", 1), ("variable", "b", 1), ("variable", "c", 1),
    ("variable", "d", 1), ("variable", "e", 1), ("variable", "i", 2), ("variable", "g", 1), ("variable", "err", 1),
    ("import", "os", 1), ("import", "system", 1), ("import", "alias", 1), ("import", "other", 1),
    ("function", "function", 1), ("class", "Klass", 1)]
x = program.lookup("x")[0]
assert x.declarations == ["body[0]:targets[0]", "body[1]:target"] and x.definition == "body[0]:targets[0]"
assert named(program, "x") == ["body[0]:targets[0]", "body[1]:target"] and program.entity_of("body[1]:target") is x
assert program.root.kind == "module" and program.root.separator == "." and program.scope_of("<module>") is program.root
assert program.lookup("c")[0].definition == "body[3]:targets[0].elts[1].elts[1].value"  # `*c`, within `(b, *c)`
assert program.lookup("d")[0].definition == "body[3]:targets[1].elements[0]"  # a list's own
assert program.lookup("i")[0].declarations == ["body[4]:target", "body[4].body[0]:targets[0]"]
assert program.lookup("err")[0].definition == "body[6].handlers[0]:name"  # a clause's own
assert [str(e.definition) for e in program.entities() if e.kind == "import"] == [
    "body[7]:names[0]", "body[7]:names[1]", "body[8]:names[0]", "body[8]:names[1]"]  # `import os.path` binds `os`
function = program.lookup("function")[0]
assert function.definition == "body[10]:name" and program.entity_of("body[10]") is function  # and its statement
assert program.lookup("Klass")[0].scope.kind == "class" and program.lookup("nothing") == []
assert refers(program, "body[3]:value.func") == [] and refers(program, "body[5]:items[0].optional_vars.elts[1].value") == []  # not bound here
assert refers(program, "body[5]:items[0].optional_vars.elts[1]") == []  # an attribute: no name is written there
print(len(listing(program)), "entities")

## P3DEF-02 · Lookup goes outward through functions to the module, past class bodies; `global` and `nonlocal`

In [ ]:
program = define("""
counter = 0
nonlocal stray
class Outer:
    size = 1
    def method(self):
        return size, counter
    class Inner:
        value = size
    squares = [size for _ in range(size)]
def outer():
    global counter, created
    counter = 1
    created = 2
    def inner():
        nonlocal later
        later = 3
        def innermost():
            nonlocal later
            return later
    def unbound():
        nonlocal missing
        missing = 4
    later = 0
def reader():
    global ghost
    return ghost
""")
assert named(program, "size") == ["body[2].body[0]:targets[0]", "body[2].body[1].body[0]:value.elts[0]",
                                  "body[2].body[2].body[0]:value", "body[2].body[3]:value.body",
                                  "body[2].body[3]:value.generators[0].iter.args[0]"]
# a method, a nested class and a comprehension's element do not see the class's names; its first iterable does
assert [refers(program, at) for at in program.named("size")] == [["Outer.size"], [], [], [], ["Outer.size"]]
assert refers(program, "body[2].body[1].body[0]:value.elts[1]") == ["counter"]
assert [e.qualified_name() for e in program.entities() if e.name in ("counter", "created", "later", "missing")] == [
    "counter", "created", "outer.later", "outer.unbound.missing"]
later = program.lookup("outer.later")[0]
assert len(program.lookup("counter")[0].declarations) == 2
assert later.declarations == ["body[3].body[3].body[1]:targets[0]", "body[3].body[5]:targets[0]"]
assert later.definition == "body[3].body[3].body[1]:targets[0]"  # in source order: `later = 3` in inner, then `later = 0`
assert refers(program, "body[3].body[3].body[2].body[1]:value") == ["outer.later"]
ghost = program.lookup("ghost")[0]
assert ghost.declarations == [] and ghost.definition is None and refers(program, "body[4].body[1]:value") == ["ghost"]
assert refers(program, "body[3].body[0]:names[0]") == ["counter"]  # the names of `global` itself
assert program.entity_of("body[3].body[0]:names[0]") is program.lookup("counter")[0]
assert program.lookup("stray")[0].parent is program.root  # `nonlocal` outside a function finds none: its own scope
assert program.lookup("Outer.Inner.value") and program.lookup("Outer.method")[0].scope.parent is program.root
print("ok")

## P3DEF-03 · Lambdas, comprehensions and assignment expressions

In [ ]:
program = define("""
def f(items):
    squares = [y := x * x for x in items if (last := x)]
    nested = {k: v for k, row in items for v in row}
    total = sum(n for n in items)
    pick = lambda item, key=items: (found := item)
    return y, last, x
x = 0
""")
assert listing(program)[:4] == [("function", "f", 1), ("parameter", "f.items", 1), ("variable", "x", 1),
                                ("variable", "x", 1)]  # the module's names first, then the function's
f = program.lookup("f")[0]
scopes = [program.scope_of(at) for at in ("body[0].body[0]:value.generators[0]", "body[0].body[1]:value.generators[0]",
                                          "body[0].body[1]:value.generators[1]",
                                          "body[0].body[2]:value.args[0].generators[0]")]
assert [s.kind for s in scopes] == ["comprehension", "comprehension", "comprehension", "comprehension"]
assert scopes[0].parent is f.scope and scopes[1] is scopes[2] and scopes[0].node == "body[0].body[0]:value"
assert program.scope_of("body[0].body[0]:value") is f.scope  # a comprehension is evaluated where it is
assert refers(program, "body[0].body[4]:value.elts[0]") == ["f.y"] and refers(program, "body[0].body[4]:value.elts[1]") == ["f.last"]
assert D.referents(program, "body[0].body[4]:value.elts[2]")[0].parent is program.root  # the comprehension's own `x` stays in it
assert refers(program, "body[0].body[0]:value.generators[0].iter") == ["f.items"]  # the first iterable is evaluated outside
assert program.scope_of("body[0].body[0]:value.generators[0].iter") is f.scope
assert program.scope_of("body[0].body[0]:value.body.target") is scopes[0]  # located where written, bound around
assert [e.kind for e in program.entities() if e.name in ("item", "key", "found")] == ["parameter", "parameter",
                                                                                      "variable"]
assert program.scope_of("body[0].body[3]:value.args.args[1].default_value") is f.scope  # defaults outside
assert program.scope_of("body[0].body[3]:value.body").kind == "lambda"
assert refers(program, "body[0].body[3]:value.body.value.target") == ["found"]
# in source order: a dict comprehension's key before its value, a lambda's parameters before its body
program = define("""
marks = {(m := 1): (m := 2) for _ in x}
reset = lambda w: (w := 0)
""")
assert program.lookup("m")[0].declarations == ["body[0]:value.key.value.target", "body[0]:value.body.value.target"]
reset = D.referents(program, "body[1]:value.args.args[0]")[0]
assert reset.kind == "parameter" and reset.declarations == ["body[1]:value.args.args[0]", "body[1]:value.body.value.target"]
print("ok")

## P3DEF-04 · Parameters, decorators, defaults, annotations, and type parameters' annotation scopes

In [ ]:
program = define("""
T = int
@decorator(T)
def generic[T: bound, *Ts, **P = default](a: T, /, b: T = T, *c: *Ts, d, **e: P) -> T:
    return a
class Box[T](Base[T], metaclass=Meta):
    item: T
type Alias[K] = dict[K, T]
type Plain = T
""")
function = program.lookup("generic")[0]
parameters = [e for e in program.entities() if e.parent is function.scope]
assert [(e.kind, e.name) for e in parameters] == [("parameter", n) for n in "abcde"]
assert [str(e.definition) for e in parameters] == ["body[1]:args.posonlyargs[0]", "body[1]:args.args[0]",
                                                   "body[1]:args.vararg", "body[1]:args.kwonlyargs[0]", "body[1]:args.kwarg"]
annotations = function.scope.parent
assert annotations.kind == "type parameters" and [e.name for e in annotations.entities()] == ["T", "Ts", "P"]
assert annotations.parent is program.root and annotations.node == "body[1]"
assert [str(e.definition) for e in annotations.entities()] == [f"body[1]:type_params[{i}]" for i in range(3)]
uses = [[e.kind for e in D.referents(program, at)] for at in program.named("T")]
assert uses == [["variable"], ["variable"],  # `T = int`, and the decorator's: the module's
                ["type parameter"],  # the type parameter itself
                ["type parameter"], ["type parameter"], ["variable"],  # the annotations' are the parameter, a default the module's
                ["type parameter"],  # the return annotation's
                ["type parameter"], ["type parameter"], ["type parameter"],  # the class's own, a base's, and in the class body
                ["variable"], ["variable"]]  # the type aliases', whose own parameters are others
assert program.scope_of("body[1]:type_params[2].default_value") is annotations
box = program.lookup("Box")[0]
assert box.scope.parent.kind == "type parameters"
assert refers(program, "body[2]:keywords[0].value") == []  # `Meta`
alias = program.lookup("Alias")[0]
assert alias.kind == "type alias" and alias.definition == "body[3]:name" and program.entity_of("body[3]") is alias
print("ok")

## P3DEF-05 · Pattern captures, and names a pattern only reads

In [ ]:
program = define("""
match command:
    case [first, *rest] | {"k": first, **rest}:
        pass
    case Point(x=0, y=py) as point if point:
        pass
    case constants.RED | _:
        pass
    case {"k": key}:
        pass
""")
assert listing(program) == [("variable", "first", 2), ("variable", "rest", 2), ("variable", "py", 1),
                            ("variable", "point", 1), ("variable", "key", 1)]
assert program.lookup("rest")[0].declarations == ["body[0].cases[0]:pattern.patterns[0].patterns[1].name",
                                                  "body[0].cases[0]:pattern.patterns[1].rest"]  # a star's, a mapping's rest
assert refers(program, "body[0].cases[1]:guard") == ["point"]
assert refers(program, "body[0].cases[2]:pattern.patterns[0].value.value") == []  # `constants`
assert refers(program, "body[0].cases[1]:pattern.pattern.kwd_attrs[0]") == []  # a keyword names an attribute
print("ok")

## P3DEF-06 · What names refer to: attributes, keywords and modules' names find nothing, and lookup takes dotted names

In [ ]:
program = define("""
import a.b
class C:
    def m(self):
        return self.m(key=1)
""")
call = "body[1].body[0].body[0]:value"
assert refers(program, f"{call}.func") == [] and refers(program, f"{call}.keywords[0]") == []  # `self.m`, `key=1`
assert refers(program, f"{call}.func.value") == ["C.m.self"]
assert refers(program, "body[0]:names[0]") == ["a"]  # `import a.b` binds `a`
assert program.lookup("C.m.self")[0].kind == "parameter" and program.lookup("C.x") == []
assert program.lookup("self", at=f"{call}.func") == program.lookup("C.m.self")
assert repr(program.lookup("C.m")[0]) == "<function C.m>"
assert D.referents(program, "body[9]:value") == [] and program.anchor("body[9]:value") is None  # not in the module
print("ok")

## P3DEF-07 · Every entity of the conformance source, every name located, and what each binds as the dialects say

Read independently of `define`, by a walk of the stored module's value, each block property (`body`, `handlers`,
`cases`, `orelse`, `finalbody`) in source order and each statement, clause or error in it through `Terms.kind_of`:
every statement, clause and error is located, its anchor's text is its place in the module's value
(`body[1].orelse[0]`), naming the statement itself, and the anchors come in the walk's order; every `variable` term's
place is located, its name written there; each assignment target, `with` target and comprehension target binds the
names mbse-expressions' `target_names` gives, and each case's pattern those its `bound_names` gives.

In [ ]:
text = (SOURCES / "python314.py").read_text(encoding="utf-8")
STORE.transaction(Parse.parse(text, "python314.py"))
started = time.perf_counter()
program = D.define(STORE, "python314.py")
elapsed = time.perf_counter() - started
entities = listing(program)
assert entities[:6] == [("import", "annotations", 1), ("import", "osp", 1), ("import", "sys", 1),
                        ("import", "sibling", 1), ("import", "one", 1), ("import", "second", 1)]
assert ("function", "Shape.__init__", 1) in entities and ("variable", "statements.counter", 3) in entities
assert ("type alias", "Pair", 1) in entities and ("variable", "patterns.origin", 1) in entities
declared = {str(at): e for e in program.entities() for at in e.declarations}


def places(value, path):
    """Every term, target or other union a value holds, with its place's text: (text, tag, value)."""
    if isinstance(value, Objects.Listed):
        for i, element in enumerate(value.values):
            yield from places(element, f"{path}[{i}]")
        return
    if not hasattr(value, "as_schema"):
        return
    record = value
    if Terms.is_term(value):
        tag, record = Terms.kind_of(value)
        yield path, tag, value
    elif isinstance(value.as_schema(), Schemas.OfUnion.Data):  # a target, or an f-string's part: its branch no step
        ((tag, record),) = Objects.properties(value).items()
        yield path, tag, value
    for name, held in Objects.properties(record).items():
        yield from places(held, f"{path}.{name}")


def bound(place):
    """The names `define` declares at a place or within it."""
    return {program.name_at(at) for at in declared if at == place or at.startswith((f"{place}.", f"{place}["))}


statements = variables = targets = patterns = 0
TARGET = re.compile(r"(:targets\[\d+\]|[:.]target|\.optional_vars)$")
BLOCKS = ("body", "handlers", "cases", "orelse", "finalbody")  # in source order
anchors = []


def blocks(node, path):
    """The statements, clauses and errors `node`'s blocks hold, each visited at its place."""
    held = Objects.properties(node)
    for block in BLOCKS:
        if block in held:
            for i, element in enumerate(held[block].values):
                visit(Terms.kind_of(element)[1], f"{path}.{block}[{i}]" if path else f"{block}[{i}]")


def visit(node, path):
    global statements, variables, targets, patterns
    statements += 1
    anchor = program.anchor(path)
    assert program.scope_of(path) is not None and str(anchor) == anchor.statement == path, path
    assert anchor.property is None and anchor.term == (), path
    anchors.append(anchor)
    for name, value in Objects.properties(node).items():
        if name in BLOCKS:
            continue
        for place, tag, term in places(value, f"{path}:{name}"):
            if tag == "variable":
                variables += 1
                assert program.scope_of(place) is not None and program.name_at(place) == Objects.properties(
                    Terms.kind_of(term)[1])["name"], place
            if TARGET.search(place):  # a statement's target, a `with`'s, a comprehension's, a named expression's
                targets += 1
                assert bound(place) == P.target_names(term), place
            if place.endswith(":pattern"):
                patterns += 1
                assert bound(place) == Patterns.bound_names(term), place
    blocks(node, path)


with STORE.transaction():
    blocks(STORE.singleton(WORKSPACE).files({"path": "python314.py"}), "")
assert program.anchor("<module>") == program.root.node == "<module>" and program.scope_of("<module>") is program.root
assert [a.order for a in anchors] == sorted(a.order for a in anchors)  # the walk's order is source order
assert (statements, variables, targets, patterns) == (151, 257, 65, 10), (statements, variables, targets, patterns)
print(len(entities), "entities;", statements, "statements,", variables, "names,", targets, "targets,", patterns,
      f"patterns; defined in {elapsed:.2f} s")

## P3DEF-08 · Anchors, and the transaction `define` reads in

An anchor is plain data: its text, its parts and its order, read with no transaction open; the text of a statement's
anchor is its place in the module's value, where the walk of the stored module finds that statement. `define` joins a
transaction open on the store, or opens one of its own; a path the workspace holds no file at is a LookupError.

In [ ]:
program = define("""
def f(a):
    return a
""")
use = program.named("a")[1]
assert (use.statement, use.property, use.term) == ("body[0].body[0]", "value", ()) and str(use) == "body[0].body[0]:value"
assert repr(use) == "Anchor('body[0].body[0]:value')" and use == program.anchor("body[0].body[0]:value") and use != 1
assert hash(use) == hash("body[0].body[0]:value") and use in {"body[0].body[0]:value"}
parameter = program.named("a")[0]
assert (parameter.property, parameter.term) == ("args", ("args", 0)) and parameter.order < use.order
assert program.root.node == "<module>" and program.anchor("body[0]").statement == "body[0]"
with STORE.transaction():
    module = STORE.singleton(WORKSPACE).files({"path": program.path})
    function = Objects.properties(module)["body"].values[0]  # body[0]
    statement = Objects.properties(Terms.kind_of(function)[1])["body"].values[0]  # body[0].body[0]
    kinds = [Terms.kind_of(x)[0] for x in (function, statement)]
    joined = D.define_file(STORE, module)  # within the caller's transaction, from the module's visitor
    with raises(ValueError, match="transactions do not nest"):  # define opens its own: called with none open
        D.define(STORE, program.path)
assert kinds == ["functiondef", "return"] and program.anchor("body[0].body[0]").statement == "body[0].body[0]"
assert program.scope_of("body[0].body[0]") is program.lookup("f")[0].scope
assert listing(joined) == listing(program) == [("function", "f", 1), ("parameter", "f.a", 1)]
assert D.referents(joined, use) == [joined.lookup("f.a")[0]]  # an anchor of one program finds its place in another
with raises(LookupError, match="absent.py"):
    D.define(STORE, "absent.py")
print("ok")

## P3DEF-09 · Invalid syntax: an `Error` binds nothing itself, the statements recovered within it bind in its scope; an `error` term's recovered terms are uses, and a `missing` term binds nothing

In [ ]:
program = define("import os\n"
                 "x = = 1\n"
                 "class C(:\n"
                 "    def m(self):\n"
                 "        return os\n"
                 "y = (z + )\n"
                 "for in w:\n"
                 "    v = 1\n")
# An `Error`'s text binds nothing; the statements recovered within it bind in the scope it stands in
assert listing(program) == [("import", "os", 1), ("function", "m", 1), ("parameter", "m.self", 1),
                            ("variable", "y", 1), ("variable", "v", 1)]
assert program.anchor("body[1]") is not None and program.scope_of("body[1]") is program.root
assert str(program.lookup("m")[0].definition) == "body[2].body[0]:name"
assert refers(program, "body[2].body[0].body[0]:value") == ["os"]  # within the recovered body, as any
# An `error` term's recovered terms are uses; a `missing` term binds nothing
assert named(program, "z") == ["body[3]:value.terms[0]"] and refers(program, "body[3]:value.terms[0]") == []
assert program.anchor("body[4]:target") is not None and named(program, "w") == ["body[4]:iter"]
print("ok")